# Notebook 22 — Shared State & Blackboard
## Agents Sharing Knowledge Through a Central Store

The **blackboard architecture**: agents read from and write to a shared knowledge store, enabling decoupled collaboration.

| Concept | Description |
|---------|-------------|
| Blackboard | Central shared knowledge store |
| Agents | Specialists that read/write to blackboard |
| Controller | Decides which agent runs next |
| Events | Notifications when data changes |

**Prerequisites:** Notebook 21 (orchestration patterns).

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. The Blackboard Architecture

Originally from AI research (Hearsay-II, 1970s), the blackboard pattern features:
- A **shared workspace** (the blackboard) holding partial solutions
- **Knowledge sources** (agents) that can read and contribute
- A **controller** that decides which agent acts next

```
         ┌─────────────────────┐
         │    BLACKBOARD       │
         │  ┌───┐ ┌───┐ ┌───┐ │
         │  │ A │ │ B │ │ C │ │  ← knowledge slots
         │  └───┘ └───┘ └───┘ │
         └──┬──────┬──────┬───┘
            │      │      │
         Agent1  Agent2  Agent3
```

**Advantages over message-passing:**
- Agents are **decoupled** — they don't need to know about each other
- New agents can be added without changing existing ones
- Shared context reduces redundant work

## 2. Building the Blackboard Class

A typed knowledge store with history, subscriptions, and access control.

In [ ]:
class BlackboardEntry:
    """A single entry on the blackboard with metadata."""
    def __init__(self, key: str, value: Any, author: str, timestamp: float):
        self.key = key
        self.value = value
        self.author = author
        self.timestamp = timestamp
        self.version = 1

    def __repr__(self):
        preview = str(self.value)[:60]
        return f"Entry({self.key}={preview}..., by={self.author}, v{self.version})"


class Blackboard:
    """Shared knowledge store for multi-agent collaboration."""

    def __init__(self, name: str = "main"):
        self.name = name
        self._store: Dict[str, BlackboardEntry] = {}
        self._history: Dict[str, List[BlackboardEntry]] = defaultdict(list)
        self._subscribers: Dict[str, List[Callable]] = defaultdict(list)
        self._access_control: Dict[str, Dict[str, set]] = {}  # key -> {readers: set, writers: set}
        self._event_log: List[Dict] = []

    def set(self, key: str, value: Any, author: str) -> None:
        """Write a value to the blackboard."""
        # Check write access
        if key in self._access_control:
            if author not in self._access_control[key].get("writers", {author}):
                self._event_log.append({"event": "ACCESS_DENIED", "key": key, "author": author, "action": "write"})
                print(f"  ⚠ ACCESS DENIED: {author} cannot write to '{key}'")
                return

        timestamp = time.time()
        entry = BlackboardEntry(key, value, author, timestamp)

        # Track version
        if key in self._store:
            entry.version = self._store[key].version + 1

        # Store and log history
        self._store[key] = entry
        self._history[key].append(entry)
        self._event_log.append({
            "event": "WRITE", "key": key, "author": author,
            "version": entry.version, "timestamp": timestamp
        })

        # Notify subscribers
        for callback in self._subscribers.get(key, []):
            callback(key, value, author)

    def get(self, key: str, reader: str = "system") -> Optional[Any]:
        """Read a value from the blackboard."""
        if key not in self._store:
            return None

        # Check read access
        if key in self._access_control:
            if reader not in self._access_control[key].get("readers", {reader}):
                self._event_log.append({"event": "ACCESS_DENIED", "key": key, "author": reader, "action": "read"})
                return None

        self._event_log.append({"event": "READ", "key": key, "reader": reader})
        return self._store[key].value

    def subscribe(self, key: str, callback: Callable):
        """Subscribe to changes on a key."""
        self._subscribers[key].append(callback)

    def set_access(self, key: str, readers: set = None, writers: set = None):
        """Set access control for a key."""
        self._access_control[key] = {
            "readers": readers or set(),
            "writers": writers or set()
        }

    def history(self, key: str) -> List[Dict]:
        """Get version history for a key."""
        return [{"version": e.version, "value": str(e.value)[:80],
                 "author": e.author, "time": e.timestamp}
                for e in self._history.get(key, [])]

    def keys(self) -> List[str]:
        return list(self._store.keys())

    def snapshot(self) -> Dict[str, str]:
        """Get a snapshot of all current values."""
        return {k: str(v.value)[:100] for k, v in self._store.items()}

    def stats(self) -> Dict[str, Any]:
        return {
            "total_keys": len(self._store),
            "total_writes": sum(len(h) for h in self._history.values()),
            "total_events": len(self._event_log),
            "authors": list(set(e.author for e in self._store.values()))
        }

# Test the blackboard
bb = Blackboard("test")
bb.set("topic", "AI Safety", "system")
bb.set("research", "AI safety involves alignment, robustness, and interpretability.", "researcher")
print(f"Blackboard '{bb.name}' — {bb.stats()}")
print(f"Topic: {bb.get('topic')}")
print(f"Research: {bb.get('research')}")
print(f"Keys: {bb.keys()}")

## 3. BlackboardAgent — Agents That Read and Write the Blackboard

Each agent:
1. **Reads** relevant keys from the blackboard for context
2. **Processes** using LLM
3. **Writes** results back to the blackboard

In [ ]:
class BlackboardAgent:
    """Agent that reads context from blackboard, processes, and writes results back."""

    def __init__(self, name: str, role: str, read_keys: List[str], write_key: str):
        self.name = name
        self.role = role
        self.read_keys = read_keys
        self.write_key = write_key
        self.runs = 0

    def run(self, blackboard: Blackboard, task: str = "") -> str:
        """Read from blackboard, process, write results back."""
        self.runs += 1

        # Read context from blackboard
        context_parts = []
        for key in self.read_keys:
            value = blackboard.get(key, reader=self.name)
            if value is not None:
                context_parts.append(f"[{key}]: {value}")

        context = "\n".join(context_parts) if context_parts else "No prior context available."

        # Build prompt
        messages = [
            {"role": "system", "content": f"You are {self.name}, a {self.role}. Use the provided context to inform your work. Be concise (3-4 sentences)."},
            {"role": "user", "content": f"Context from shared knowledge:\n{context}\n\nTask: {task}"}
        ]

        t0 = time.time()
        response = generate(messages, max_new_tokens=250)
        elapsed = time.time() - t0

        # Write results to blackboard
        blackboard.set(self.write_key, response, self.name)

        print(f"  [{self.name}] Read: {self.read_keys} → Write: '{self.write_key}' ({elapsed:.1f}s)")
        return response

    def __repr__(self):
        return f"BBAgent({self.name}, reads={self.read_keys}, writes='{self.write_key}')"

# Create blackboard agents
researcher = BlackboardAgent("Researcher", "research analyst who gathers information",
                              read_keys=["topic", "task"], write_key="research")
analyst = BlackboardAgent("Analyst", "analytical expert who identifies patterns and insights",
                           read_keys=["topic", "research"], write_key="analysis")
synthesizer = BlackboardAgent("Synthesizer", "expert writer who combines information into clear summaries",
                               read_keys=["research", "analysis"], write_key="synthesis")

print("Created BlackboardAgents:")
for a in [researcher, analyst, synthesizer]:
    print(f"  {a}")

## 4. Experiment — Three Agents Solve a Problem via Blackboard

**Workflow:**
1. System writes the topic to blackboard
2. Researcher reads topic → writes research
3. Analyst reads topic + research → writes analysis
4. Synthesizer reads research + analysis → writes synthesis

Agents communicate **only through the blackboard** — no direct message passing.

In [ ]:
# Fresh blackboard for the experiment
board = Blackboard("experiment_1")

# Set the task
topic = "The impact of large language models on software engineering practices"
board.set("topic", topic, "system")
board.set("task", "Provide a comprehensive analysis of this topic", "system")

print("=" * 70)
print("BLACKBOARD EXPERIMENT: 3 Agents Collaborate")
print("=" * 70)
print(f"Topic: {topic}\n")

# Agent 1: Research
print("Phase 1: Research")
researcher.run(board, "Research this topic thoroughly. Identify key trends and evidence.")

# Agent 2: Analysis
print("\nPhase 2: Analysis")
analyst.run(board, "Analyze the research findings. Identify patterns, implications, and gaps.")

# Agent 3: Synthesis
print("\nPhase 3: Synthesis")
synthesizer.run(board, "Synthesize the research and analysis into a coherent summary with conclusions.")

In [ ]:
# Examine blackboard state
print("=" * 70)
print("BLACKBOARD STATE AFTER EXPERIMENT")
print("=" * 70)

for key in board.keys():
    value = board.get(key)
    print(f"\n[{key}] (by {board._store[key].author}, v{board._store[key].version}):")
    print(f"  {str(value)[:250]}")

print(f"\n{'=' * 70}")
print("BLACKBOARD STATISTICS")
print("=" * 70)
stats = board.stats()
for k, v in stats.items():
    print(f"  {k}: {v}")

## 5. Event-Driven Activation — Agents React to Changes

Instead of running agents in a fixed order, let them **activate when relevant data changes**.
The blackboard's subscribe mechanism triggers agents automatically.

In [ ]:
class EventDrivenBlackboard(Blackboard):
    """Blackboard with event-driven agent activation."""

    def __init__(self, name: str = "event_driven"):
        super().__init__(name)
        self._agent_triggers: Dict[str, List[Tuple['BlackboardAgent', str]]] = defaultdict(list)
        self._activation_log: List[Dict] = []

    def register_trigger(self, watch_key: str, agent: 'BlackboardAgent', task: str):
        """Register an agent to activate when a key changes."""
        self._agent_triggers[watch_key].append((agent, task))

    def set(self, key: str, value: Any, author: str) -> None:
        """Write value and trigger registered agents."""
        super().set(key, value, author)

        # Trigger agents watching this key
        if key in self._agent_triggers:
            for agent, task in self._agent_triggers[key]:
                if agent.name != author:  # prevent self-triggering
                    print(f"    ⚡ Triggered: {agent.name} (watching '{key}')")
                    self._activation_log.append({
                        "trigger_key": key, "agent": agent.name,
                        "caused_by": author
                    })
                    agent.run(self, task)


# Set up event-driven system
eboard = EventDrivenBlackboard("event_experiment")

# Create agents
e_researcher = BlackboardAgent("E-Researcher", "research specialist",
                                read_keys=["topic"], write_key="research")
e_analyst = BlackboardAgent("E-Analyst", "analysis specialist",
                             read_keys=["topic", "research"], write_key="analysis")
e_writer = BlackboardAgent("E-Writer", "summary writer",
                            read_keys=["research", "analysis"], write_key="summary")

# Register triggers: analyst activates when research changes,
# writer activates when analysis changes
eboard.register_trigger("research", e_analyst, "Analyze the new research findings.")
eboard.register_trigger("analysis", e_writer, "Write a summary of the research and analysis.")

print("Event-driven blackboard configured:")
print("  research changes → triggers E-Analyst")
print("  analysis changes → triggers E-Writer")
print()

In [ ]:
# Start the cascade by writing topic and then research
print("=" * 70)
print("EVENT-DRIVEN CASCADE")
print("=" * 70)

eboard.set("topic", "Quantum computing applications in drug discovery", "system")
print("\nWriting initial research (this should trigger a cascade)...\n")

# This single write should cascade: research → analysis → summary
e_researcher.run(eboard, "Research quantum computing in drug discovery.")

print(f"\n{'=' * 70}")
print("ACTIVATION LOG")
print("=" * 70)
for entry in eboard._activation_log:
    print(f"  Key '{entry['trigger_key']}' changed by {entry['caused_by']} → activated {entry['agent']}")

print(f"\nFinal summary:")
print(eboard.get("summary", "system")[:400] if eboard.get("summary", "system") else "No summary generated")

## 6. Conflict Resolution — When Agents Disagree

What happens when two agents write to the **same key**? We need resolution strategies:
1. **Last-write-wins** (default)
2. **Version-based merge** — keep both versions
3. **LLM-mediated resolution** — use LLM to reconcile

In [ ]:
class ConflictResolvingBlackboard(Blackboard):
    """Blackboard with conflict resolution strategies."""

    STRATEGIES = ["last_write_wins", "keep_all", "llm_merge"]

    def __init__(self, name: str, strategy: str = "last_write_wins"):
        super().__init__(name)
        assert strategy in self.STRATEGIES
        self.strategy = strategy
        self.conflicts: List[Dict] = []

    def set(self, key: str, value: Any, author: str) -> None:
        if key in self._store and self._store[key].author != author:
            # Conflict detected
            existing = self._store[key]
            self.conflicts.append({
                "key": key,
                "existing_author": existing.author,
                "existing_value": str(existing.value)[:100],
                "new_author": author,
                "new_value": str(value)[:100],
                "strategy": self.strategy
            })
            print(f"  ⚠ Conflict on '{key}': {existing.author} vs {author}")

            if self.strategy == "last_write_wins":
                super().set(key, value, author)

            elif self.strategy == "keep_all":
                # Store as list
                if isinstance(existing.value, list):
                    merged = existing.value + [{"author": author, "value": value}]
                else:
                    merged = [
                        {"author": existing.author, "value": existing.value},
                        {"author": author, "value": value}
                    ]
                super().set(key, merged, "system_merge")

            elif self.strategy == "llm_merge":
                merged = self._llm_resolve(key, existing.value, existing.author, value, author)
                super().set(key, merged, "llm_merge")
        else:
            super().set(key, value, author)

    def _llm_resolve(self, key: str, val1: Any, author1: str, val2: Any, author2: str) -> str:
        messages = [
            {"role": "system", "content": "You reconcile conflicting information. Combine both perspectives into a coherent merged version. Be concise."},
            {"role": "user", "content": f"Key: {key}\n[{author1}]: {val1}\n[{author2}]: {val2}\n\nMerge these into one coherent entry."}
        ]
        return generate(messages, max_new_tokens=200)


# Test all three strategies
print("=" * 70)
print("CONFLICT RESOLUTION STRATEGIES")
print("=" * 70)

strategies = ["last_write_wins", "keep_all", "llm_merge"]
for strategy in strategies:
    print(f"\n--- Strategy: {strategy} ---")
    bb = ConflictResolvingBlackboard(f"conflict_{strategy}", strategy)

    # Two agents write to the same key
    bb.set("conclusion", "LLMs will replace most coding tasks within 5 years.", "Optimist")
    bb.set("conclusion", "LLMs will augment but not replace programmers.", "Skeptic")

    result = bb.get("conclusion")
    if isinstance(result, list):
        for item in result:
            print(f"  [{item['author']}]: {str(item['value'])[:80]}")
    else:
        print(f"  Result: {str(result)[:150]}")

    print(f"  Conflicts detected: {len(bb.conflicts)}")

## 7. Blackboard vs Message-Passing — Comparison

Let's solve the **same task** using both approaches and compare.

In [ ]:
# === Message-Passing Approach ===
class MessagePassingAgent:
    """Agent that communicates via direct messages."""
    def __init__(self, name: str, role: str):
        self.name = name
        self.role = role
        self.inbox: List[Dict] = []
        self.outbox: List[Dict] = []

    def receive(self, message: Dict):
        self.inbox.append(message)

    def process(self, task: str) -> str:
        context = "\n".join([f"From {m['from']}: {m['content']}" for m in self.inbox])
        messages = [
            {"role": "system", "content": f"You are {self.name}, a {self.role}. Be concise (2-3 sentences)."},
            {"role": "user", "content": f"Messages received:\n{context}\n\nTask: {task}" if context else f"Task: {task}"}
        ]
        response = generate(messages, max_new_tokens=200)
        return response

    def send(self, to_agent: 'MessagePassingAgent', content: str):
        msg = {"from": self.name, "content": content}
        to_agent.receive(msg)
        self.outbox.append({**msg, "to": to_agent.name})


def run_message_passing(topic: str) -> Dict[str, Any]:
    """Solve a task using message-passing."""
    t0 = time.time()
    mp_researcher = MessagePassingAgent("MP-Researcher", "researcher")
    mp_analyst = MessagePassingAgent("MP-Analyst", "analyst")
    mp_writer = MessagePassingAgent("MP-Writer", "writer")

    # Step 1: researcher works
    research = mp_researcher.process(f"Research: {topic}")
    mp_researcher.send(mp_analyst, research)
    mp_researcher.send(mp_writer, research)

    # Step 2: analyst works
    analysis = mp_analyst.process("Analyze the research you received.")
    mp_analyst.send(mp_writer, analysis)

    # Step 3: writer synthesizes
    synthesis = mp_writer.process("Write a summary combining all inputs.")

    return {
        "approach": "message_passing",
        "result": synthesis,
        "time": round(time.time() - t0, 2),
        "messages_sent": sum(len(a.outbox) for a in [mp_researcher, mp_analyst, mp_writer])
    }


def run_blackboard(topic: str) -> Dict[str, Any]:
    """Solve the same task using blackboard."""
    t0 = time.time()
    bb = Blackboard("comparison")
    bb.set("topic", topic, "system")

    bb_researcher = BlackboardAgent("BB-Researcher", "researcher", ["topic"], "research")
    bb_analyst = BlackboardAgent("BB-Analyst", "analyst", ["topic", "research"], "analysis")
    bb_writer = BlackboardAgent("BB-Writer", "writer", ["research", "analysis"], "synthesis")

    bb_researcher.run(bb, f"Research: {topic}")
    bb_analyst.run(bb, "Analyze the research.")
    bb_writer.run(bb, "Write a summary combining research and analysis.")

    return {
        "approach": "blackboard",
        "result": bb.get("synthesis"),
        "time": round(time.time() - t0, 2),
        "bb_writes": bb.stats()["total_writes"]
    }

print("✓ Both approaches defined. Ready for comparison.")

In [ ]:
# Run comparison
topic = "How renewable energy is transforming the global economy"

print("=" * 70)
print("BLACKBOARD vs MESSAGE-PASSING COMPARISON")
print("=" * 70)
print(f"Topic: {topic}\n")

print("--- Running Message-Passing approach ---")
mp_result = run_message_passing(topic)
print(f"  Time: {mp_result['time']}s | Messages: {mp_result['messages_sent']}")

print("\n--- Running Blackboard approach ---")
bb_result = run_blackboard(topic)
print(f"  Time: {bb_result['time']}s | BB writes: {bb_result['bb_writes']}")

print(f"\n{'=' * 70}")
print("COMPARISON")
print("=" * 70)
print(f"""
                  │ Message-Passing │ Blackboard
──────────────────┼─────────────────┼───────────
Time              │ {mp_result['time']:>13}s │ {bb_result['time']:>8}s
Coupling          │ Direct (tight)  │ Indirect (loose)
Add new agent     │ Rewire messages │ Just read/write keys
History/Audit     │ Per-agent inbox │ Central log
Conflict handling │ N/A             │ Built-in strategies
""")

print("Message-Passing result preview:")
print(f"  {str(mp_result['result'])[:200]}")
print(f"\nBlackboard result preview:")
print(f"  {str(bb_result['result'])[:200]}")

## 8. Key Takeaways

### Blackboard Architecture
1. **Shared state** reduces coupling — agents don't need to know about each other
2. **History tracking** provides full audit trail of how knowledge evolved
3. **Event-driven activation** enables reactive, cascade-style workflows
4. **Access control** prevents unauthorized reads/writes to sensitive keys

### Conflict Resolution
- **Last-write-wins**: Simple but risks data loss
- **Keep-all**: Preserves everything but may accumulate noise
- **LLM-mediated merge**: Smart resolution but adds latency

### When to Use What
| Use Blackboard When | Use Message-Passing When |
|---------------------|--------------------------|
| Many agents, loose coupling | Few agents, tight coordination |
| Need audit trail | Low-latency required |
| Dynamic agent addition | Fixed agent topology |
| Shared context is large | Messages are small |

### Next Steps
→ **Notebook 23**: Swarm intelligence — many simple agents with emergent behavior
→ **Notebook 24**: Agent safety and guardrails